In [1]:
import sys
sys.path.append("..")
import numpy as np
import pickle
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

from lm_conf.default_utils.custom_types import OrganisedOutputs, PromptCollection
from lm_conf.models.vllm_model import vLLMModel
from lm_conf.post_processing.metrics import BetaDistribution
from lm_conf.post_processing.metrics import dAUROC, dECE_equal_mass, dECE_equal_width

/home/ivan/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = vLLMModel({"name": "meta-llama/Llama-3.1-8b-Instruct", "repeat": 3, "max_model_len": 512})

In [3]:
questions = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Who wrote '1984'?"
]

continuations = [
    [" Paris", " London", " Berlin"],
    [" 3", " 4", " 5"],
    [" George Orwell", " Aldous Huxley", " Ray Bradbury"]
]

prompt_collection = PromptCollection(
    questions=["What is the capital of France?", "What is 2 + 2?", "Who wrote '1984'?"],
    answer_keys=["Paris", "4", "George Orwell"],
    context_texts=["What is the capital of France?", "What is 2 + 2?", "Who wrote '1984'?"],
    continuation_texts={q: a for q, a in zip(questions, continuations)}
)

In [4]:
outputs = model.run_generation(prompt_collection)

INFO 12-25 00:00:50 [utils.py:253] non-default args: {'max_model_len': 512, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8b-Instruct'}


INFO 12-25 00:00:53 [model.py:631] Resolved architecture: LlamaForCausalLM
INFO 12-25 00:00:53 [model.py:1745] Using max model len 512


2025-12-25 00:00:53,111	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 12-25 00:00:53 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=2184601) INFO 12-25 00:00:55 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='meta-llama/Llama-3.1-8b-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8b-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  2.14it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.97it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  2.81it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.46it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.40it/s]
(EngineCore_DP0 pid=2184601) 


(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:00 [default_loader.py:314] Loading weights took 1.74 seconds
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:01 [gpu_model_runner.py:3338] Model loading took 14.9889 GiB memory and 3.812670 seconds
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:05 [backends.py:631] Using cache directory: /home/ivan/.cache/vllm/torch_compile_cache/602d62bee1/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:05 [backends.py:647] Dynamo bytecode transform time: 3.85 s
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:07 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.911 s
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:08 [monitor.py:34] torch.compile takes 5.76 s in total
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:09 [gpu_worker.py:359] Available KV cache memory: 5.02 GiB
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:10 [kv_cache_utils.py:1229] GPU KV cache size: 41,104 tokens
(E

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 24.22it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 29.13it/s]


(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:13 [gpu_model_runner.py:4244] Graph capturing finished in 4 secs, took 0.56 GiB
(EngineCore_DP0 pid=2184601) INFO 12-25 00:01:13 [core.py:250] init engine (profile, create kv cache, warmup model) took 12.63 seconds
INFO 12-25 00:01:16 [llm.py:352] Supported tasks: ['generate']
INFO 12-25 00:01:21 [chat_utils.py:557] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 9/9 [00:02<00:00,  3.79it/s, est. speed input: 159.20 toks/s, output: 68.23 toks/s]


In [5]:
for i in range(len(questions)):
    print(outputs[i].output_texts)
    print(outputs[i].output_logprobs)
    print(outputs[i].output_tokens)

['The capital of France is Paris.', '2 + 2 = 4.', "The book '1984' was written by the English author George Orwell. His full name was Eric Arthur Blair and he used the pseudonym George Orwell for his literary work. It was first published in 1949."]
[[-0.00020644917094614357, -5.602820692729438e-06, -5.602820692729438e-06, -1.5497195136049413e-06, -1.1920928244535389e-07, -4.0649541915627196e-05, -0.00011955977242905647], [-0.47433599829673767, -5.447716102935374e-05, -1.1920928244535389e-07, -4.291525328881107e-06, -0.20379024744033813, -2.3841855067985307e-07, -7.867782187531702e-06, -0.16023750603199005], [-0.6752108335494995, -2.268486499786377, -0.22763222455978394, -2.932505594799295e-05, 0.0, -0.013058030046522617, -0.00928819552063942, -0.00025769727653823793, -5.364403477869928e-06, -1.4616386890411377, -0.26401466131210327, -0.06393992900848389, -0.0408753976225853, -2.109982233378105e-05, -0.08379069715738297, -4.556061267852783, -0.12119507044553757, -0.0006559127941727638, 